# NHANES Depression Risk Prediction

## Notebook 02: Data Preprocessing

This notebook prepares the merged NHANES 2017–2018 dataset for exploratory analysis and machine learning.

### Preprocessing objectives

1. Load and validate the merged dataset.
2. Identify duplicate records and incorrect values.
3. Recode special NHANES response codes.
4. Calculate the PHQ-9 score.
5. Create the binary elevated-depression-risk target.
6. Analyze missing values.
7. Evaluate potential outliers.
8. Remove target leakage.
9. Save the processed dataset.

In [1]:
# Import the libraries required for data preprocessing
from pathlib import Path

import numpy as np
import pandas as pd

# Configure pandas to display more columns when inspecting the dataset
pd.set_option("display.max_columns", 50)

# Define the location of the merged dataset
input_file = Path("../data/interim/nhanes_2017_2018_merged.csv")

# Stop with a clear message if the input file cannot be found
if not input_file.exists():
    raise FileNotFoundError(f"Dataset not found: {input_file}")

# Load the merged NHANES dataset
df = pd.read_csv(input_file)

# Display initial validation information
print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate participant IDs:", df["SEQN"].duplicated().sum())

# Preview the first five records
df.head()

Dataset loaded successfully!
Dataset shape: (5533, 42)
Duplicate rows: 0
Duplicate participant IDs: 0


,SEQN,DPQ010,DPQ020,DPQ030,DPQ040,DPQ050,DPQ060,DPQ070,DPQ080,DPQ090,RIAGENDR,RIDAGEYR,RIDRETH3,DMDEDUC2,DMDMARTL,INDFMPIR,WTMEC2YR,SDMVPSU,SDMVSTRA,SLD012,SLD013,SLQ030,SLQ040,SLQ050,SLQ120,BMXBMI,BMXWAIST,PAQ605,PAQ620,PAQ635,PAQ650,PAQ665,PAD680,SMQ020,SMQ040,SMD650,ALQ111,ALQ121,ALQ130,ALQ142,ALQ151,ALQ170
0,93705.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,66.0,4.0,2.0,3.0,0.82,8338.419786,2.0,145.0,8.0,8.0,2.0,0.0,2.0,0.0,31.7,101.8,2.0,2.0,2.0,2.0,1.0,300.0,1.0,3.0,NaN,1.0,7.0,1.0,0.0,2.0,5.0
1,93706.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,18.0,6.0,NaN,NaN,NaN,8723.439814,2.0,134.0,10.5,11.5,1.0,0.0,2.0,1.0,21.5,79.3,2.0,2.0,1.0,2.0,1.0,240.0,2.0,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN
2,93708.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,66.0,6.0,1.0,1.0,1.63,14372.488765,2.0,138.0,8.0,8.0,9.0,0.0,2.0,2.0,23.7,88.2,2.0,2.0,2.0,2.0,1.0,120.0,2.0,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN
3,93709.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,75.0,4.0,4.0,2.0,0.41,12277.556662,1.0,136.0,7.0,6.5,1.0,0.0,2.0,1.0,38.9,113.0,2.0,1.0,2.0,2.0,2.0,600.0,1.0,1.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN
4,93711.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,56.0,6.0,5.0,1.0,5.00,12390.919724,2.0,134.0,7.0,9.0,2.0,1.0,1.0,3.0,21.3,86.6,2.0,2.0,1.0,1.0,1.0,420.0,2.0,NaN,NaN,1.0,5.0,1.0,0.0,2.0,0.0


## 1. Initial Data Audit

This section examines each variable’s data type, number of unique values, and missing-value percentage before any cleaning or transformation is applied.

In [2]:
# Create a column-level data-quality summary
data_quality = pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "unique_values": df.nunique(dropna=True),
    "missing_count": df.isna().sum(),
    "missing_percentage": df.isna().mean().mul(100).round(2)
})

# Sort variables from the highest to the lowest missing percentage
data_quality = data_quality.sort_values(
    by="missing_percentage",
    ascending=False
)

# Display the complete data-quality summary
data_quality

,data_type,unique_values,missing_count,missing_percentage
SMD650,float64,28,4573,82.65
SMQ040,float64,3,3301,59.66
ALQ170,float64,21,2046,36.98
ALQ142,float64,13,2038,36.83
ALQ130,float64,16,2038,36.83
ALQ151,float64,4,989,17.87
ALQ121,float64,13,988,17.86
INDFMPIR,float64,431,734,13.27
DPQ090,float64,6,443,8.01
DPQ060,float64,6,442,7.99


## 2. PHQ-9 Score and Target Creation

The nine depression-screening items are scored from 0 to 3. Responses outside this range represent missing, refused, or unknown answers and are converted to missing values.

A PHQ-9 score is calculated only when all nine items are available:

- **0:** PHQ-9 score below 10
- **1:** PHQ-9 score of 10 or higher

The resulting target represents elevated depressive-symptom risk, not a clinical diagnosis.

In [3]:
# Identify the nine PHQ-9 depression-screening variables
phq9_columns = [
    "DPQ010", "DPQ020", "DPQ030",
    "DPQ040", "DPQ050", "DPQ060",
    "DPQ070", "DPQ080", "DPQ090"
]

# Create a working copy so the original imported data remains unchanged
df_clean = df.copy()

# Keep only valid PHQ-9 responses between 0 and 3
# Refused, unknown, and other invalid responses become missing values
for column in phq9_columns:
    df_clean[column] = df_clean[column].where(
        df_clean[column].isin([0, 1, 2, 3]),
        np.nan
    )

# Count the number of completed PHQ-9 questions for each participant
df_clean["phq9_completed_items"] = (
    df_clean[phq9_columns].notna().sum(axis=1)
)

# Calculate the PHQ-9 score only when all nine questions are complete
df_clean["phq9_score"] = df_clean[phq9_columns].sum(
    axis=1,
    min_count=9
)

# Create the binary target while preserving missing PHQ-9 scores
df_clean["elevated_depression_risk"] = np.where(
    df_clean["phq9_score"].notna(),
    (df_clean["phq9_score"] >= 10).astype(int),
    np.nan
)

# Summarize target availability
print(
    "Complete PHQ-9 records:",
    df_clean["phq9_score"].notna().sum()
)
print(
    "Incomplete PHQ-9 records:",
    df_clean["phq9_score"].isna().sum()
)

# Display the target distribution
target_distribution = (
    df_clean["elevated_depression_risk"]
    .value_counts(dropna=False)
    .rename_axis("target")
    .to_frame("count")
)

target_distribution["percentage"] = (
    target_distribution["count"]
    .div(len(df_clean))
    .mul(100)
    .round(2)
)

target_distribution

Complete PHQ-9 records: 5068
Incomplete PHQ-9 records: 465


,count,percentage
target,,
0.0,4609,83.3
NaN,465,8.4
1.0,459,8.3


### 2.1 Target Eligibility and Class Balance

Participants with incomplete PHQ-9 scores cannot be assigned a reliable target and are excluded from supervised modeling.

The remaining dataset is imbalanced, with elevated depressive-symptom risk representing approximately 9% of complete records. Therefore, stratified data splitting and imbalance-handling methods will be used during model training.

In [4]:
# Keep only participants with a complete and valid target
eligible_df = df_clean.dropna(
    subset=["elevated_depression_risk"]
).copy()

# Convert the target from float to integer
eligible_df["elevated_depression_risk"] = (
    eligible_df["elevated_depression_risk"].astype("int8")
)

# Calculate the number of excluded participants
excluded_records = len(df_clean) - len(eligible_df)

# Calculate class counts and percentages among eligible participants
class_balance = (
    eligible_df["elevated_depression_risk"]
    .value_counts()
    .sort_index()
    .rename_axis("target")
    .to_frame("count")
)

class_balance["percentage"] = (
    class_balance["count"]
    .div(len(eligible_df))
    .mul(100)
    .round(2)
)

# Add readable class descriptions
class_balance["description"] = [
    "PHQ-9 below 10",
    "PHQ-9 of 10 or higher"
]

# Display validation results
print("Eligible participants:", len(eligible_df))
print("Excluded incomplete records:", excluded_records)
print("Eligible dataset shape:", eligible_df.shape)

class_balance

Eligible participants: 5068
Excluded incomplete records: 465
Eligible dataset shape: (5068, 45)


,count,percentage,description
target,,,
0,4609,90.94,PHQ-9 below 10
1,459,9.06,PHQ-9 of 10 or higher


## 3. Cleaning Special Response Codes

NHANES uses special numeric codes for responses such as "Refused" and
"Don't know." These values are not genuine measurements or categories and
are therefore converted to missing values.

The replacement rules are defined separately for each variable according
to the official NHANES 2017–2018 codebooks. Valid values such as sleep
duration boundary values are retained.

In [5]:
# Import NumPy for its standard missing-value representation
import numpy as np

# Create a separate working dataset for preprocessing
model_df = eligible_df.copy()

# Define special NHANES response codes for demographic variables
demographic_missing_codes = {
    "DMDEDUC2": [7, 9],    # Refused, Don't know
    "DMDMARTL": [77, 99],  # Refused, Don't know
}

# Define special NHANES response codes for sleep variables
sleep_missing_codes = {
    "SLQ030": [7, 9],  # Refused, Don't know
    "SLQ040": [7, 9],  # Refused, Don't know
    "SLQ050": [7, 9],  # Refused, Don't know
    "SLQ120": [7, 9],  # Refused, Don't know
}

# Combine the verified replacement rules
special_missing_codes = {
    **demographic_missing_codes,
    **sleep_missing_codes,
}

# Store a summary of the replacements
replacement_summary = []

# Replace each variable's special response codes with missing values
for column, codes in special_missing_codes.items():

    # Count special responses before replacing them
    replacement_count = model_df[column].isin(codes).sum()

    # Convert the special response codes to NaN
    model_df[column] = model_df[column].replace(codes, np.nan)

    # Record the result for documentation
    replacement_summary.append({
        "variable": column,
        "codes_replaced": str(codes),
        "values_replaced": replacement_count,
        "missing_after_cleaning": model_df[column].isna().sum(),
    })

# Convert the summary into a readable table
replacement_summary_df = pd.DataFrame(replacement_summary)

print("Special response code cleaning completed.")
print("Dataset shape:", model_df.shape)

replacement_summary_df

Special response code cleaning completed.
Dataset shape: (5068, 45)


,variable,codes_replaced,values_replaced,missing_after_cleaning
0,DMDEDUC2,"[7, 9]",6,249
1,DMDMARTL,"[77, 99]",5,248
2,SLQ030,"[7, 9]",370,370
3,SLQ040,"[7, 9]",288,288
4,SLQ050,"[7, 9]",3,3
5,SLQ120,"[7, 9]",8,8


In [6]:
# Define special response codes for physical-activity variables
activity_missing_codes = {
    "PAQ605": [7, 9],       # Refused, Don't know
    "PAQ620": [7, 9],       # Refused, Don't know
    "PAQ635": [7, 9],       # Refused, Don't know
    "PAQ650": [7, 9],       # Refused, Don't know
    "PAQ665": [7, 9],       # Refused, Don't know
    "PAD680": [7777, 9999], # Refused, Don't know
}

# Define special response codes for smoking variables
smoking_missing_codes = {
    "SMQ020": [7, 9],       # Refused, Don't know
    "SMQ040": [7, 9],       # Refused, Don't know
    "SMD650": [777, 999],   # Refused, Don't know
}

# Define special response codes for alcohol-use variables
alcohol_missing_codes = {
    "ALQ111": [7, 9],       # Refused, Don't know
    "ALQ121": [77, 99],     # Refused, Don't know
    "ALQ130": [777, 999],   # Refused, Don't know
    "ALQ142": [77, 99],     # Refused, Don't know
    "ALQ151": [7, 9],       # Refused, Don't know
    "ALQ170": [777, 999],   # Refused, Don't know
}

# Combine the additional variable-specific cleaning rules
additional_missing_codes = {
    **activity_missing_codes,
    **smoking_missing_codes,
    **alcohol_missing_codes,
}

# Create a list for documenting the replacements
additional_replacement_summary = []

# Replace special response codes with missing values
for column, codes in additional_missing_codes.items():

    # Count the special codes before replacement
    replacement_count = model_df[column].isin(codes).sum()

    # Replace only the verified special codes
    model_df[column] = model_df[column].replace(codes, np.nan)

    # Save the cleaning result
    additional_replacement_summary.append({
        "variable": column,
        "codes_replaced": str(codes),
        "values_replaced": replacement_count,
        "missing_after_cleaning": model_df[column].isna().sum(),
    })

# Display a professional cleaning summary
additional_replacement_summary_df = pd.DataFrame(
    additional_replacement_summary
)

print("Additional special response code cleaning completed.")
print("Dataset shape:", model_df.shape)

additional_replacement_summary_df

Additional special response code cleaning completed.
Dataset shape: (5068, 45)


,variable,codes_replaced,values_replaced,missing_after_cleaning
0,PAQ605,"[7, 9]",4,4
1,PAQ620,"[7, 9]",3,3
2,PAQ635,"[7, 9]",0,0
3,PAQ650,"[7, 9]",0,0
4,PAQ665,"[7, 9]",0,0
5,PAD680,"[7777, 9999]",27,36
6,SMQ020,"[7, 9]",0,0
7,SMQ040,"[7, 9]",0,2990
8,SMD650,"[777, 999]",1,4176
9,ALQ111,"[7, 9]",0,3


In [7]:
# Restore ALQ142 from the original eligible dataset because
# valid categories 7 and 9 were accidentally replaced
model_df["ALQ142"] = eligible_df["ALQ142"].copy()

# Count the correct special response codes
alq142_replacement_count = model_df["ALQ142"].isin([77, 99]).sum()

# Replace only Refused (77) and Don't know (99)
model_df["ALQ142"] = model_df["ALQ142"].replace(
    [77, 99],
    np.nan
)

# Confirm the correction
print("ALQ142 correction completed.")
print("Special codes replaced:", alq142_replacement_count)
print("Missing values after correction:",
      model_df["ALQ142"].isna().sum())

# Verify that valid categories 7 and 9 remain in the dataset
model_df["ALQ142"].value_counts(
    dropna=False
).sort_index()

ALQ142 correction completed.
Special codes replaced: 8
Missing values after correction: 1598


ALQ142
0.0     2040
1.0       34
2.0       26
3.0       61
4.0      103
5.0      105
6.0      214
7.0      141
8.0       96
9.0      235
10.0     415
NaN     1598
Name: count, dtype: int64

## 4. Target Leakage Prevention and Modeling Dataset

The elevated depression-risk target was calculated from the nine PHQ-9
questionnaire items. Therefore, the individual PHQ-9 items, calculated
PHQ-9 score, and completion count must be excluded from the predictor
variables.

Participant identifiers and NHANES survey-design variables are also
separated from the machine-learning predictors. This prevents identity
information and survey-design metadata from influencing model predictions.

In [9]:
# Define the nine PHQ-9 questionnaire variables used to create the target
phq9_columns = [
    "DPQ010",
    "DPQ020",
    "DPQ030",
    "DPQ040",
    "DPQ050",
    "DPQ060",
    "DPQ070",
    "DPQ080",
    "DPQ090",
]

# Define additional columns that must not be used as predictors
non_predictor_columns = [
    "SEQN",                 # Participant identifier
    "phq9_completed_items", # Used during target construction
    "phq9_score",           # Directly determines the target
    "WTMEC2YR",             # NHANES examination sample weight
    "SDMVPSU",              # Survey-design primary sampling unit
    "SDMVSTRA",             # Survey-design stratum
]

# Preserve participant identifiers for record tracking
participant_ids = model_df["SEQN"].copy()

# Preserve the survey-design variables separately for possible
# population-level analysis
survey_design_df = model_df[
    ["SEQN", "WTMEC2YR", "SDMVPSU", "SDMVSTRA"]
].copy()

# Remove leakage, identifiers, and survey-design metadata
modeling_df = model_df.drop(
    columns=phq9_columns + non_predictor_columns
).copy()

# Confirm that none of the leakage variables remain
remaining_leakage_columns = [
    column
    for column in phq9_columns + non_predictor_columns
    if column in modeling_df.columns
]

print("Modeling dataset created successfully.")
print("Original cleaned shape:", model_df.shape)
print("Modeling dataset shape:", modeling_df.shape)
print("Number of predictors:", modeling_df.shape[1] - 1)
print("Remaining leakage columns:", remaining_leakage_columns)

# Display the first five records
modeling_df.head()

Modeling dataset created successfully.
Original cleaned shape: (5068, 45)
Modeling dataset shape: (5068, 30)
Number of predictors: 29
Remaining leakage columns: []


,RIAGENDR,RIDAGEYR,RIDRETH3,DMDEDUC2,DMDMARTL,INDFMPIR,SLD012,SLD013,SLQ030,SLQ040,SLQ050,SLQ120,BMXBMI,BMXWAIST,PAQ605,PAQ620,PAQ635,PAQ650,PAQ665,PAD680,SMQ020,SMQ040,SMD650,ALQ111,ALQ121,ALQ130,ALQ142,ALQ151,ALQ170,elevated_depression_risk
0,2.0,66.0,4.0,2.0,3.0,0.82,8.0,8.0,2.0,0.0,2.0,0.0,31.7,101.8,2.0,2.0,2.0,2.0,1.0,300.0,1.0,3.0,NaN,1.0,7.0,1.0,0.0,2.0,5.0,0
1,1.0,18.0,6.0,NaN,NaN,NaN,10.5,11.5,1.0,0.0,2.0,1.0,21.5,79.3,2.0,2.0,1.0,2.0,1.0,240.0,2.0,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,0
2,2.0,66.0,6.0,1.0,1.0,1.63,8.0,8.0,NaN,0.0,2.0,2.0,23.7,88.2,2.0,2.0,2.0,2.0,1.0,120.0,2.0,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,0
4,1.0,56.0,6.0,5.0,1.0,5.00,7.0,9.0,2.0,1.0,1.0,3.0,21.3,86.6,2.0,2.0,1.0,1.0,1.0,420.0,2.0,NaN,NaN,1.0,5.0,1.0,0.0,2.0,0.0,0
5,1.0,18.0,1.0,NaN,NaN,0.76,7.5,9.0,1.0,1.0,2.0,2.0,19.7,72.0,1.0,1.0,2.0,1.0,1.0,120.0,1.0,2.0,2.0,1.0,0.0,NaN,NaN,2.0,NaN,0


## 5. Missing-Value Analysis After Cleaning

Missing values are recalculated after removing special response codes and
target-leakage variables. Some missingness results from NHANES questionnaire
eligibility rules and skip patterns rather than data-quality errors.

For example, detailed smoking questions are only asked of applicable
participants, and some alcohol questions depend on answers to earlier
questions. Missing values will therefore be handled through feature
engineering and model preprocessing instead of automatically deleting
participants.

In [10]:
# Calculate the number and percentage of missing values
# in the final modeling dataset
modeling_missing_summary = pd.DataFrame({
    "data_type": modeling_df.dtypes.astype(str),
    "unique_values": modeling_df.nunique(dropna=True),
    "missing_count": modeling_df.isna().sum(),
    "missing_percentage": (
        modeling_df.isna().mean() * 100
    ).round(2),
})

# Add a broad explanation of the likely missingness source
structural_missing_variables = [
    "DMDEDUC2",
    "DMDMARTL",
    "SMQ040",
    "SMD650",
    "ALQ121",
    "ALQ130",
    "ALQ142",
    "ALQ151",
    "ALQ170",
]

modeling_missing_summary["missingness_note"] = np.where(
    modeling_missing_summary.index.isin(
        structural_missing_variables
    ),
    "May include eligibility or survey skip-pattern missingness",
    "General item nonresponse or measurement missingness",
)

# Keep only variables containing at least one missing value
modeling_missing_summary = (
    modeling_missing_summary[
        modeling_missing_summary["missing_count"] > 0
    ]
    .sort_values(
        by="missing_percentage",
        ascending=False
    )
)

print(
    "Variables with missing values:",
    modeling_missing_summary.shape[0]
)

modeling_missing_summary

Variables with missing values: 22


,data_type,unique_values,missing_count,missing_percentage,missingness_note
SMD650,float64,26,4176,82.40,May include eligibility or survey skip-pattern...
SMQ040,float64,3,2990,59.00,May include eligibility or survey skip-pattern...
ALQ170,float64,19,1783,35.18,May include eligibility or survey skip-pattern...
ALQ142,float64,11,1598,31.53,May include eligibility or survey skip-pattern...
ALQ130,float64,14,1594,31.45,May include eligibility or survey skip-pattern...
INDFMPIR,float64,429,630,12.43,General item nonresponse or measurement missin...
ALQ151,float64,2,572,11.29,May include eligibility or survey skip-pattern...
ALQ121,float64,11,571,11.27,May include eligibility or survey skip-pattern...
SLQ030,float64,4,370,7.30,General item nonresponse or measurement missin...
SLQ040,float64,4,288,5.68,General item nonresponse or measurement missin...



## 6. Feature Engineering for Smoking Status

The smoking questionnaire uses conditional questions. Participants who
reported never smoking 100 cigarettes were not asked detailed current-smoking
questions. Consequently, missing values in these variables do not necessarily
represent unknown information.

A consolidated smoking-status feature is created with four categories:

- 0: Never smoker
- 1: Former smoker
- 2: Some-day smoker
- 3: Daily smoker

For never and former smokers, cigarettes smoked per day is logically set to
zero. The original smoking-status variables are then removed because their
information is represented by the engineered feature.

In [11]:
# Create a new dataset for feature engineering
engineered_df = modeling_df.copy()

# Record missing cigarettes-per-day values before feature engineering
cigarettes_missing_before = engineered_df["SMD650"].isna().sum()

# Define conditions using lifetime and current smoking responses
smoking_conditions = [
    engineered_df["SMQ020"].eq(2),
    (
        engineered_df["SMQ020"].eq(1)
        & engineered_df["SMQ040"].eq(3)
    ),
    (
        engineered_df["SMQ020"].eq(1)
        & engineered_df["SMQ040"].eq(2)
    ),
    (
        engineered_df["SMQ020"].eq(1)
        & engineered_df["SMQ040"].eq(1)
    ),
]

# Assign an interpretable smoking-status category
smoking_categories = [
    0,  # Never smoker
    1,  # Former smoker
    2,  # Some-day smoker
    3,  # Daily smoker
]

engineered_df["smoking_status"] = np.select(
    smoking_conditions,
    smoking_categories,
    default=np.nan,
)

# Never and former smokers currently smoke zero cigarettes per day
non_current_smoker_mask = (
    engineered_df["smoking_status"].isin([0, 1])
)

engineered_df.loc[
    non_current_smoker_mask,
    "SMD650"
] = 0

# Rename the numeric variable to make its meaning clearer
engineered_df = engineered_df.rename(
    columns={"SMD650": "cigarettes_per_day"}
)

# Remove the original variables now represented by smoking_status
engineered_df = engineered_df.drop(
    columns=["SMQ020", "SMQ040"]
)

# Create readable labels for reviewing the engineered feature
smoking_status_labels = {
    0: "Never smoker",
    1: "Former smoker",
    2: "Some-day smoker",
    3: "Daily smoker",
}

smoking_status_summary = (
    engineered_df["smoking_status"]
    .map(smoking_status_labels)
    .value_counts(dropna=False)
    .rename_axis("smoking_status")
    .reset_index(name="count")
)

# Calculate percentages
smoking_status_summary["percentage"] = (
    smoking_status_summary["count"]
    / len(engineered_df)
    * 100
).round(2)

print("Smoking feature engineering completed.")
print("Dataset shape:", engineered_df.shape)
print(
    "Missing cigarettes-per-day values before:",
    cigarettes_missing_before,
)
print(
    "Missing cigarettes-per-day values after:",
    engineered_df["cigarettes_per_day"].isna().sum(),
)

smoking_status_summary

Smoking feature engineering completed.
Dataset shape: (5068, 29)
Missing cigarettes-per-day values before: 4176
Missing cigarettes-per-day values after: 13


,smoking_status,count,percentage
0,Never smoker,2990,59.00
1,Former smoker,1173,23.15
2,Daily smoker,713,14.07
3,Some-day smoker,192,3.79


## 7. Feature Engineering for Alcohol Use

Several alcohol-use questions are conditionally administered. Participants
who never consumed alcohol, or who did not drink during the previous year,
were not asked detailed consumption questions.

To distinguish structural missingness from unknown responses, an alcohol-use
status feature is created:

- 0: Lifetime abstainer
- 1: No alcohol use during the past year
- 2: Current alcohol user

Missing consumption values are set to zero only when earlier responses
confirm that the participant did not drink. Observed answers are preserved.

In [12]:
# Record alcohol missingness before feature engineering
alcohol_columns_before = [
    "ALQ111",
    "ALQ121",
    "ALQ130",
    "ALQ142",
    "ALQ151",
    "ALQ170",
]

alcohol_missing_before = (
    engineered_df[alcohol_columns_before]
    .isna()
    .sum()
)

# Participants who never consumed alcohol did not drink
# during the previous 12 months
lifetime_abstainer_mask = engineered_df["ALQ111"].eq(2)

engineered_df.loc[
    lifetime_abstainer_mask
    & engineered_df["ALQ121"].isna(),
    "ALQ121",
] = 0

# Create an interpretable alcohol-use status
alcohol_conditions = [
    engineered_df["ALQ111"].eq(2),
    (
        engineered_df["ALQ111"].eq(1)
        & engineered_df["ALQ121"].eq(0)
    ),
    (
        engineered_df["ALQ111"].eq(1)
        & engineered_df["ALQ121"].between(1, 10)
    ),
]

alcohol_categories = [
    0,  # Lifetime abstainer
    1,  # No alcohol use during the past year
    2,  # Current alcohol user
]

engineered_df["alcohol_use_status"] = np.select(
    alcohol_conditions,
    alcohol_categories,
    default=np.nan,
)

# Identify participants with confirmed no alcohol use
# during the previous year
no_past_year_alcohol_mask = (
    engineered_df["ALQ121"].eq(0)
)

# For confirmed non-drinkers, structurally missing consumption
# values logically represent zero consumption
for column in ["ALQ130", "ALQ142", "ALQ170"]:
    engineered_df.loc[
        no_past_year_alcohol_mask
        & engineered_df[column].isna(),
        column,
    ] = 0

# Convert lifetime heavy daily drinking into a binary feature
# 1 = Yes, 0 = No
engineered_df["ever_heavy_daily_drinking"] = np.select(
    [
        engineered_df["ALQ151"].eq(1),
        engineered_df["ALQ151"].eq(2),
        lifetime_abstainer_mask,
    ],
    [1, 0, 0],
    default=np.nan,
)

# Rename alcohol variables using descriptive names
engineered_df = engineered_df.rename(
    columns={
        "ALQ121": "alcohol_frequency",
        "ALQ130": "average_drinks_per_day",
        "ALQ142": "binge_drinking_frequency",
        "ALQ170": "binge_drinking_episodes_30d",
    }
)

# Remove the original variables represented by engineered features
engineered_df = engineered_df.drop(
    columns=["ALQ111", "ALQ151"]
)

# Create readable alcohol-status labels for validation
alcohol_status_labels = {
    0: "Lifetime abstainer",
    1: "No alcohol use in past year",
    2: "Current alcohol user",
}

alcohol_status_summary = (
    engineered_df["alcohol_use_status"]
    .map(alcohol_status_labels)
    .value_counts(dropna=False)
    .rename_axis("alcohol_use_status")
    .reset_index(name="count")
)

# Calculate class percentages
alcohol_status_summary["percentage"] = (
    alcohol_status_summary["count"]
    / len(engineered_df)
    * 100
).round(2)

# Compare missingness after feature engineering
alcohol_columns_after = [
    "alcohol_frequency",
    "average_drinks_per_day",
    "binge_drinking_frequency",
    "binge_drinking_episodes_30d",
    "ever_heavy_daily_drinking",
    "alcohol_use_status",
]

alcohol_missing_after = (
    engineered_df[alcohol_columns_after]
    .isna()
    .sum()
)

print("Alcohol feature engineering completed.")
print("Dataset shape:", engineered_df.shape)

display(alcohol_status_summary)

print("Missing values after feature engineering:")
alcohol_missing_after.to_frame(name="missing_count")

Alcohol feature engineering completed.
Dataset shape: (5068, 29)


,alcohol_use_status,count,percentage
0,Current alcohol user,3475,68.57
1,No alcohol use in past year,1022,20.17
2,Lifetime abstainer,564,11.13
3,NaN,7,0.14


Missing values after feature engineering:


,missing_count
alcohol_frequency,7
average_drinks_per_day,8
binge_drinking_frequency,12
binge_drinking_episodes_30d,197
ever_heavy_daily_drinking,8
alcohol_use_status,7


## 8. Demographic Eligibility Handling

The NHANES adult education and marital-status variables are released only
for participants aged 20 years or older. Therefore, missing education and
marital-status values among participants aged 18–19 represent questionnaire
ineligibility rather than ordinary missing data.

A separate category coded as 0 is assigned to these participants:

- Education level 0: Not applicable — participant under age 20
- Marital status 0: Not applicable — participant under age 20

Missing values among participants aged 20 or older are retained for later
imputation.

In [13]:
# Identify participants younger than 20
under_20_mask = engineered_df["RIDAGEYR"] < 20

# Record missing values before eligibility handling
demographic_missing_before = engineered_df[
    ["DMDEDUC2", "DMDMARTL"]
].isna().sum()

# Assign category 0 only when the participant is under 20
# and the education value is structurally missing
engineered_df.loc[
    under_20_mask
    & engineered_df["DMDEDUC2"].isna(),
    "DMDEDUC2",
] = 0

# Assign category 0 only when the participant is under 20
# and marital status is structurally missing
engineered_df.loc[
    under_20_mask
    & engineered_df["DMDMARTL"].isna(),
    "DMDMARTL",
] = 0

# Calculate missing values after eligibility handling
demographic_missing_after = engineered_df[
    ["DMDEDUC2", "DMDMARTL"]
].isna().sum()

# Create a comparison table for documentation
demographic_eligibility_summary = pd.DataFrame({
    "missing_before": demographic_missing_before,
    "missing_after": demographic_missing_after,
})

demographic_eligibility_summary["values_resolved"] = (
    demographic_eligibility_summary["missing_before"]
    - demographic_eligibility_summary["missing_after"]
)

print("Demographic eligibility handling completed.")
print("Participants younger than 20:", under_20_mask.sum())
print("Dataset shape:", engineered_df.shape)

demographic_eligibility_summary

Demographic eligibility handling completed.
Participants younger than 20: 243
Dataset shape: (5068, 29)


,missing_before,missing_after,values_resolved
DMDEDUC2,249,6,243
DMDMARTL,248,5,243


## 9. Sleep and Physical-Activity Feature Engineering

Weekday and weekend sleep duration are combined into:

- Average weekly sleep duration
- Weekend sleep difference

The weighted weekly average uses five weekdays and two weekend days.

Physical-activity responses are converted from the NHANES coding system
(1 = Yes, 2 = No) into binary indicators (1 = Active, 0 = Not active).
An additional feature counts the number of activity domains reported by
each participant.

In [14]:
# ---------------------------------------------------------
# Sleep feature engineering
# ---------------------------------------------------------

# Calculate the weighted average sleep duration:
# five weekdays and two weekend days
engineered_df["average_sleep_hours"] = (
    engineered_df["SLD012"] * 5
    + engineered_df["SLD013"] * 2
) / 7

# Calculate how much sleep changes on weekends
# Positive values indicate more sleep on weekends
engineered_df["weekend_sleep_difference"] = (
    engineered_df["SLD013"]
    - engineered_df["SLD012"]
)

# Remove the original sleep-duration variables because their
# information is represented by the engineered features
engineered_df = engineered_df.drop(
    columns=["SLD012", "SLD013"]
)

# ---------------------------------------------------------
# Physical-activity feature engineering
# ---------------------------------------------------------

# Map the original NHANES variables to descriptive names
activity_variable_names = {
    "PAQ605": "vigorous_work_activity",
    "PAQ620": "moderate_work_activity",
    "PAQ635": "walking_or_bicycling",
    "PAQ650": "vigorous_recreation",
    "PAQ665": "moderate_recreation",
}

# Convert 1 = Yes and 2 = No into binary indicators
for original_column, new_column in activity_variable_names.items():
    engineered_df[new_column] = (
        engineered_df[original_column]
        .map({
            1: 1,  # Yes
            2: 0,  # No
        })
    )

# Remove the original NHANES activity columns
engineered_df = engineered_df.drop(
    columns=list(activity_variable_names.keys())
)

# List the newly created binary activity features
activity_columns = list(activity_variable_names.values())

# Count the number of activity domains reported by each participant
# A score is produced only when all five activity responses are present
engineered_df["active_domain_count"] = (
    engineered_df[activity_columns]
    .sum(
        axis=1,
        min_count=len(activity_columns),
    )
)

# Rename sedentary time using a descriptive feature name
engineered_df = engineered_df.rename(
    columns={
        "PAD680": "sedentary_minutes_per_day",
    }
)

# ---------------------------------------------------------
# Validate the engineered features
# ---------------------------------------------------------

engineered_feature_summary = engineered_df[
    [
        "average_sleep_hours",
        "weekend_sleep_difference",
        "sedentary_minutes_per_day",
        "active_domain_count",
    ]
].describe().T.round(2)

print("Sleep and activity feature engineering completed.")
print("Dataset shape:", engineered_df.shape)
print(
    "Missing average sleep values:",
    engineered_df["average_sleep_hours"].isna().sum(),
)
print(
    "Missing activity-count values:",
    engineered_df["active_domain_count"].isna().sum(),
)

engineered_feature_summary

Sleep and activity feature engineering completed.
Dataset shape: (5068, 30)
Missing average sleep values: 53
Missing activity-count values: 7


,count,mean,std,min,25%,50%,75%,max
average_sleep_hours,5015.0,7.79,1.50,2.0,7.0,7.86,8.71,14.0
weekend_sleep_difference,5015.0,0.69,1.64,-9.0,0.0,0.00,1.50,10.0
sedentary_minutes_per_day,5032.0,332.27,199.11,0.0,180.0,300.00,480.00,1320.0
active_domain_count,5061.0,1.58,1.28,0.0,1.0,1.00,2.00,5.0


The engineered values look reasonable. The extreme sleep and sedentary values are possible NHANES boundary values, so we should identify them statistically without deleting them automatically.

## 10. Outlier Analysis

Potential outliers are identified for continuous variables using the
interquartile-range (IQR) method. A value is flagged when it falls below
Q1 − 1.5 × IQR or above Q3 + 1.5 × IQR.

An IQR flag does not necessarily indicate an error. Health measurements and
behavioral variables may contain legitimate extreme values. Therefore,
outliers are documented but not removed automatically. Any transformation
or clipping will later be fitted only on training data to prevent data
leakage.

In [15]:
# Define continuous variables appropriate for IQR outlier analysis
continuous_features = [
    "RIDAGEYR",
    "INDFMPIR",
    "BMXBMI",
    "BMXWAIST",
    "cigarettes_per_day",
    "average_drinks_per_day",
    "binge_drinking_episodes_30d",
    "average_sleep_hours",
    "weekend_sleep_difference",
    "sedentary_minutes_per_day",
]

# Create a list to store the outlier statistics
outlier_results = []

# Calculate IQR boundaries separately for each feature
for column in continuous_features:

    # Exclude missing values when calculating quartiles
    valid_values = engineered_df[column].dropna()

    # Calculate the first and third quartiles
    q1 = valid_values.quantile(0.25)
    q3 = valid_values.quantile(0.75)

    # Calculate the interquartile range
    iqr = q3 - q1

    # Calculate the standard IQR boundaries
    lower_bound = q1 - (1.5 * iqr)
    upper_bound = q3 + (1.5 * iqr)

    # Identify values outside the IQR boundaries
    outlier_mask = (
        (valid_values < lower_bound)
        | (valid_values > upper_bound)
    )

    # Count the potential outliers
    outlier_count = outlier_mask.sum()

    # Store the results
    outlier_results.append({
        "feature": column,
        "minimum": valid_values.min(),
        "q1": q1,
        "q3": q3,
        "maximum": valid_values.max(),
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "outlier_count": outlier_count,
        "outlier_percentage": (
            outlier_count / len(valid_values) * 100
        ),
    })

# Convert the results into a formatted table
outlier_summary_df = pd.DataFrame(outlier_results)

# Round numeric results for readability
numeric_columns = [
    "minimum",
    "q1",
    "q3",
    "maximum",
    "lower_bound",
    "upper_bound",
    "outlier_percentage",
]

outlier_summary_df[numeric_columns] = (
    outlier_summary_df[numeric_columns].round(2)
)

# Sort variables by their percentage of potential outliers
outlier_summary_df = outlier_summary_df.sort_values(
    by="outlier_percentage",
    ascending=False,
).reset_index(drop=True)

print("IQR outlier analysis completed.")
print("No observations were removed.")

outlier_summary_df

IQR outlier analysis completed.
No observations were removed.


,feature,minimum,q1,q3,maximum,lower_bound,upper_bound,outlier_count,outlier_percentage
0,binge_drinking_episodes_30d,0.0,0.00,0.00,30.0,0.00,0.00,868,17.82
1,cigarettes_per_day,0.0,0.00,0.00,60.0,0.00,0.00,892,17.65
2,weekend_sleep_difference,-9.0,0.00,1.50,10.0,-2.25,3.75,357,7.12
3,average_drinks_per_day,0.0,0.00,2.00,15.0,-3.00,5.00,274,5.42
4,average_sleep_hours,2.0,7.00,8.71,14.0,4.43,11.29,189,3.77
5,BMXBMI,14.8,24.62,33.60,86.2,11.16,47.06,141,2.81
6,BMXWAIST,56.4,88.00,111.00,169.5,53.50,145.50,59,1.21
7,sedentary_minutes_per_day,0.0,180.00,480.00,1320.0,-270.00,930.00,42,0.83
8,RIDAGEYR,18.0,33.00,64.00,80.0,-13.50,110.50,0,0.00
9,INDFMPIR,0.0,1.18,4.08,5.0,-3.17,8.43,0,0.00


The outlier analysis is correct. The high percentages for cigarettes and binge drinking happen because most values are zero, making both quartiles zero. These are valid non-use values—not errors.

## 11. Final Validation and Data Export

The preprocessing stage concludes with validation of the engineered dataset.
Potential statistical outliers are retained because they fall within plausible
NHANES response ranges.

Missing-value imputation, categorical encoding, scaling, and resampling are
not performed at this stage. These operations will be fitted only on the
training data through machine-learning pipelines to prevent data leakage.

The processed dataset retains the participant identifier for traceability,
but this identifier will be excluded from model training.

In [16]:
# Create the final preprocessed dataset
final_preprocessed_df = engineered_df.copy()

# Restore the participant identifier for traceability
# It will be removed again before model training
final_preprocessed_df.insert(
    0,
    "SEQN",
    participant_ids.to_numpy(),
)

# Define the target variable
target_column = "elevated_depression_risk"

# Validate the final dataset
validation_results = {
    "number_of_rows": len(final_preprocessed_df),
    "number_of_columns": final_preprocessed_df.shape[1],
    "number_of_predictors": (
        final_preprocessed_df.shape[1] - 2
    ),  # Excludes SEQN and target
    "unique_participants": (
        final_preprocessed_df["SEQN"].nunique()
    ),
    "duplicate_participant_ids": (
        final_preprocessed_df["SEQN"].duplicated().sum()
    ),
    "missing_target_values": (
        final_preprocessed_df[target_column].isna().sum()
    ),
    "positive_target_cases": (
        final_preprocessed_df[target_column].sum()
    ),
}

validation_summary_df = pd.DataFrame(
    validation_results.items(),
    columns=["validation_check", "result"],
)

# Ensure critical validation rules are satisfied
assert validation_results["number_of_rows"] == 5068
assert validation_results["unique_participants"] == 5068
assert validation_results["duplicate_participant_ids"] == 0
assert validation_results["missing_target_values"] == 0

print("Final dataset validation completed successfully.")

validation_summary_df

Final dataset validation completed successfully.


,validation_check,result
0,number_of_rows,5068
1,number_of_columns,31
2,number_of_predictors,29
3,unique_participants,5068
4,duplicate_participant_ids,0
5,missing_target_values,0
6,positive_target_cases,459


In [17]:
# Create the processed-data folder if it does not already exist
processed_folder = Path("../data/processed")
processed_folder.mkdir(
    parents=True,
    exist_ok=True,
)

# Define output file locations
processed_data_file = (
    processed_folder
    / "nhanes_2017_2018_preprocessed.csv"
)

survey_design_file = (
    processed_folder
    / "nhanes_2017_2018_survey_design.csv"
)

# Save the preprocessed modeling dataset
final_preprocessed_df.to_csv(
    processed_data_file,
    index=False,
)

# Save survey weights and design variables separately
survey_design_df.to_csv(
    survey_design_file,
    index=False,
)

print("Preprocessed dataset saved successfully.")
print("Modeling data:", processed_data_file)
print("Survey-design data:", survey_design_file)
print("Saved shape:", final_preprocessed_df.shape)

Preprocessed dataset saved successfully.
Modeling data: ../data/processed/nhanes_2017_2018_preprocessed.csv
Survey-design data: ../data/processed/nhanes_2017_2018_survey_design.csv
Saved shape: (5068, 31)
